# GitHub Integration

Claude Code has an **official GitHub integration** that runs Claude inside **GitHub Actions**. Two workflows: **mention support** (in issues/PRs) and **automatic PR reviews**.


## Setup

Run **`/install-github-app`** in Claude. It walks you through:

1. Install the **Claude Code app** on GitHub
2. Add your **API key** (stored as a repo secret)
3. Auto-generate a **pull request** with the workflow files

Merging that PR adds two GitHub Actions under **`.github/workflows/`**.

> **Currency note:** setup is still `/install-github-app`, but auth now supports either an **API key** *or* a **Claude subscription via an OAuth token** secret, and the underlying action (`anthropics/claude-code-action`) has evolved — check current docs for the exact inputs. The concepts below are unchanged.

## The two default actions

**Mention action** — write `@claude` in any issue or PR. Claude will:
- analyze the request and create a task plan
- execute it with full access to your codebase
- respond with results directly in the issue/PR

**Pull request action** — when a PR is opened, Claude automatically:
- reviews the proposed changes
- analyzes the impact of the modifications
- posts a detailed report on the PR

## Customizing the workflows

After merging, edit the workflow YAML to fit your project.

**Add environment/project setup** before Claude runs:
```yaml
- name: Project Setup
  run: |
    npm run setup
    npm run dev:daemon
```

**Custom instructions** — give Claude project context:
```yaml
custom_instructions: |
  The project is already set up with all dependencies installed.
  The server is already running at localhost:3000. Logs from it
  are being written to logs.txt. If needed, you can query the
  db with the 'sqlite3' cli. If needed, use the mcp__playwright
  set of tools to launch a browser and interact with the app.
```

**MCP server config** — extra capabilities in CI:
```yaml
mcp_config: |
  {
    "mcpServers": {
      "playwright": {
        "command": "npx",
        "args": [
          "@playwright/mcp@latest",
          "--allowed-origins",
          "localhost:3000;cdn.tailwindcss.com;esm.sh"
        ]
      }
    }
  }
```

## Tool permissions (important)

In GitHub Actions you must **explicitly list every allowed tool** — there's **no permission shortcut** like local dev. Each tool from each MCP server has to be named individually:

```yaml
allowed_tools: "Bash(npm:*),Bash(sqlite3:*),mcp__playwright__browser_snapshot,mcp__playwright__browser_click,..."
```

So enabling an MCP server in CI means enumerating each of its tools you want Claude to use.

## Best practices

- Start with the **default workflows**, customize gradually.
- Use **custom instructions** for project-specific context (how it's set up, how to run/inspect it).
- Be **explicit about tool permissions**, especially with MCP servers.
- **Test with simple tasks** before complex ones.
- Tailor the extra setup steps/config to your project's needs.

The GitHub integration turns Claude from a local assistant into an **automated team member** — handling `@claude` tasks, reviewing PRs, and posting insights right in your GitHub workflow.

That completes the **Getting hands on** section. Next in the course: the **Hooks and the SDK** section, then the **quiz** + summary.